# PAPER-EXP-01C — Frozen Clean Internal Test

**Final locked candidate:** fixed 50/50 fusion of the accepted **B0-300 TTA7**
and **B3-300 TTA7** three-seed families.

This notebook performs **no training and no model selection**.

It evaluates the already-locked system on the exact **971-image / 749-lesion**
clean TEST partition (manifest SHA256 `5c6f3cdca8ae60e200d6e5f5f1d619b3f08df71a2dadab95cdad7948a2834eec`).

The TEST was previously observed by LITE-GEN 04B, so this is correctly described
as a **frozen repeated clean internal test benchmark**, not a pristine never-seen
project-level test.

The Results ZIP automatically creates the complete paper package and a paired
same-test comparison against the historical LITE-GEN 04B predictions.


In [1]:
from pathlib import Path

SCRIPT = 'from __future__ import annotations\n\nimport base64\nimport contextlib\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nimport platform\nimport shutil\nimport sys\nimport zipfile\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\nimport sklearn\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_recall_fscore_support,\n    recall_score,\n    roc_auc_score,\n    roc_curve,\n)\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torchvision\nfrom torch.utils.data import Dataset, DataLoader\nfrom torchvision import models, transforms\nfrom tqdm.auto import tqdm\n\n\nCODE_VERSION = "PAPEREXP01C_FROZEN_CLEAN_INTERNAL_TEST_FIXED_B0B3_FUSION_2026-08-23_v1"\n\nINPUT_ROOT = Path("/kaggle/input")\nWORKING_ROOT = Path("/kaggle/working")\nOUT = WORKING_ROOT / "PAPEREXP01C_Frozen_Clean_Internal_Test"\nFIG = OUT / "figures"\nSRC = OUT / "figure_source_data"\nTAB = OUT / "tables"\nAUDIT = OUT / "audit"\nfor p in (OUT, FIG, SRC, TAB, AUDIT):\n    p.mkdir(parents=True, exist_ok=True)\n\nCLASS_NAMES = ("MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC")\nNUM_CLASSES = 7\nMEL_INDEX = 0\nIMAGE_SIZE = 300\nBATCH_SIZE = 32\nNUM_WORKERS = 4\n\nTEST_MANIFEST_SHA = "5c6f3cdca8ae60e200d6e5f5f1d619b3f08df71a2dadab95cdad7948a2834eec"\nLG03_RESULTS_ZIP_SHA = "982dc6b1509b28f50f4940db2ac110002971ca0d7ac3aa1c78b07582442fb24f"\n\nB0_RECOVERY_ZIP_SHA = "778060755a8be16d8ab7bb21055d50a6f0a641a763f19caf30da399096192668"\nB3_RECOVERY_ZIP_SHA = "2a25768706d4bd7ee7ce6f6982ceedd6b0b97de936cf3739f68cdba7e2bc9ae8"\n\nB0_HASHES = {\'20260820\': \'30efcb4f88cfddd4f6ece227d920fbab27caf38e821f2e442941510b199bfb39\', \'20260821\': \'a79eef838354a6613d75b7aa455d643ccd0f3309b23a31f177c3169f723575cb\', \'20260822\': \'f258083d9bb36858ad96ce2d2fd8ba54a0834dc5b8f38dcbb1ee762783509587\'}\nB3_HASHES = {\'20260820\': \'59a23d41c48cb1ccbf9184e9cf67c115717775560283bb4a1da95f14698dc296\', \'20260821\': \'723176d2c08b759b9a8c2cbbfe245840ad844f1a77da748948c75c09afc51d69\', \'20260822\': \'2dc2cf4d28b6f5b8ccdb61a173aff25ba80ab2e0c794dfdfafcb104a87d89a4d\'}\n\nOLD_04B_PRED_SHA = "5a2c5ca2a39319bec601d1fb6e999c21b909f6f829a9bb76a82643104c1b0a4a"\nOLD_04B_PRED_GZIP_B64 = """..."""\n\nOLD_04B_METRICS = {\n    "accuracy": 0.8527291452111225,\n    "macro_f1": 0.6713432549644158,\n    "balanced_accuracy": 0.6912816493603928,\n    "melanoma_recall": 0.638095238095238,\n    "macro_auc": 0.9696825962107176,\n}\nOLD_04B_CI = {\n    "accuracy": (0.8233037968703247, 0.8807243179966993),\n    "macro_f1": (0.6082072216805173, 0.7356766716665047),\n    "balanced_accuracy": (0.6349177279134739, 0.7719564728707679),\n    "melanoma_recall": (0.5150765076507651, 0.7556201550387597),\n    "macro_auc": (0.9591125559867325, 0.9785318049423484),\n}\n\nBOOTSTRAP_ITERS = 2000\nBOOTSTRAP_SEED = 20260821\nPROB_COLS = [f"prob_{c}" for c in CLASS_NAMES]\nIMG_EXTS = {".jpg", ".jpeg", ".png"}\n\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha256_file(path: Path, block_size: int = 1 << 20) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(block_size), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef locate_exact_file_by_hash(expected_hash: str, suffixes=None):\n    suffixes = {s.lower() for s in (suffixes or [])}\n    for p in INPUT_ROOT.rglob("*"):\n        if not p.is_file():\n            continue\n        if suffixes and p.suffix.lower() not in suffixes:\n            continue\n        try:\n            if sha256_file(p) == expected_hash:\n                return p\n        except Exception:\n            continue\n    return None\n\n\ndef load_test_manifest():\n    for p in INPUT_ROOT.rglob("litegen03_test_manifest.csv"):\n        if not p.is_file():\n            continue\n        raw = p.read_bytes()\n        if sha256_bytes(raw) == TEST_MANIFEST_SHA:\n            print("[TEST MANIFEST]", p)\n            return pd.read_csv(io.BytesIO(raw)), str(p)\n\n    zp = locate_exact_file_by_hash(LG03_RESULTS_ZIP_SHA, suffixes={".zip"})\n    if zp is not None:\n        with zipfile.ZipFile(zp) as z:\n            raw = z.read("litegen03_test_manifest.csv")\n        if sha256_bytes(raw) != TEST_MANIFEST_SHA:\n            raise RuntimeError("Frozen TEST manifest SHA256 mismatch.")\n        print("[TEST MANIFEST FROM ZIP]", zp)\n        return pd.read_csv(io.BytesIO(raw)), f"{zp}::litegen03_test_manifest.csv"\n\n    raise FileNotFoundError(\n        "Attach the exact LITEGEN03 Clean Split dataset containing "\n        "litegen03_test_manifest.csv (SHA256 guard failed/not found)."\n    )\n\n\ndef find_locked_checkpoints():\n    expected = {\n        **{f"B0_{seed}": digest for seed, digest in B0_HASHES.items()},\n        **{f"B3_{seed}": digest for seed, digest in B3_HASHES.items()},\n    }\n    reverse = {digest: key for key, digest in expected.items()}\n    found = {}\n    provenance = {}\n\n    # Kaggle dataset may expose checkpoint files directly.\n    for p in INPUT_ROOT.rglob("*"):\n        if not p.is_file() or p.suffix.lower() not in {".pth", ".pt", ".ckpt"}:\n            continue\n        try:\n            digest = sha256_file(p)\n        except Exception:\n            continue\n        key = reverse.get(digest)\n        if key and key not in found:\n            found[key] = p\n            provenance[key] = {\n                "source": str(p),\n                "sha256": digest,\n                "mode": "direct_file",\n            }\n\n    if set(found) == set(expected):\n        return found, provenance\n\n    # Otherwise inspect ZIP members and extract only exact locked checkpoints.\n    extract_dir = OUT / "locked_checkpoints"\n    extract_dir.mkdir(parents=True, exist_ok=True)\n\n    for zp in INPUT_ROOT.rglob("*.zip"):\n        if not zp.is_file():\n            continue\n        try:\n            with zipfile.ZipFile(zp) as z:\n                for info in z.infolist():\n                    if not info.filename.lower().endswith((".pth", ".pt", ".ckpt")):\n                        continue\n                    data = z.read(info)\n                    digest = sha256_bytes(data)\n                    key = reverse.get(digest)\n                    if key is None or key in found:\n                        continue\n                    p = extract_dir / f"{key}.pth"\n                    p.write_bytes(data)\n                    found[key] = p\n                    provenance[key] = {\n                        "source": f"{zp}::{info.filename}",\n                        "source_zip_sha256": sha256_file(zp),\n                        "sha256": digest,\n                        "mode": "zip_member",\n                    }\n                    print("[LOCKED CHECKPOINT]", key, digest)\n        except zipfile.BadZipFile:\n            continue\n        if set(found) == set(expected):\n            break\n\n    missing = sorted(set(expected) - set(found))\n    if missing:\n        raise FileNotFoundError(\n            "Could not locate all six frozen PAPER-EXP-01A/01B checkpoints. "\n            f"Missing: {missing}. Attach both Recovery datasets."\n        )\n\n    for key, p in found.items():\n        if sha256_file(p) != expected[key]:\n            raise RuntimeError(f"Checkpoint hash changed after extraction: {key}")\n\n    return found, provenance\n\n\ndef index_images(test_df: pd.DataFrame):\n    required = set(test_df["image_id"].astype(str))\n    found = {}\n    for p in INPUT_ROOT.rglob("*"):\n        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:\n            continue\n        stem = p.stem\n        if stem in required and stem not in found:\n            found[stem] = p\n    missing = sorted(required - set(found))\n    if missing:\n        raise FileNotFoundError(\n            f"Missing {len(missing)} frozen TEST images. Examples: {missing[:15]}"\n        )\n    return found\n\n\nclass TestDataset(Dataset):\n    def __init__(self, frame, image_index):\n        self.frame = frame.reset_index(drop=True)\n        self.image_index = image_index\n        self.tf = transforms.Compose([\n            transforms.Resize(\n                (IMAGE_SIZE, IMAGE_SIZE),\n                interpolation=transforms.InterpolationMode.BILINEAR,\n            ),\n            transforms.ToTensor(),\n            transforms.Normalize(\n                [0.485, 0.456, 0.406],\n                [0.229, 0.224, 0.225],\n            ),\n        ])\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, idx):\n        row = self.frame.iloc[idx]\n        with Image.open(self.image_index[str(row["image_id"])]) as im:\n            im = ImageOps.exif_transpose(im).convert("RGB")\n            x = self.tf(im)\n        return x, int(row["label"]), str(row["image_id"]), str(row["lesion_id"])\n\n\ndef normalize_state_keys(state):\n    keys = list(state)\n    if keys and all(str(k).startswith("module.") for k in keys):\n        return {str(k)[7:]: v for k, v in state.items()}\n    if keys and all(str(k).startswith("_orig_mod.") for k in keys):\n        return {str(k)[10:]: v for k, v in state.items()}\n    return state\n\n\ndef state_dict_from_checkpoint(obj):\n    if isinstance(obj, dict):\n        for key in ("model_state_dict", "state_dict", "model", "ema_state_dict", "model_ema_state_dict"):\n            value = obj.get(key)\n            if isinstance(value, dict) and value and all(torch.is_tensor(v) for v in value.values()):\n                return value\n        if obj and all(torch.is_tensor(v) for v in obj.values()):\n            return obj\n    raise RuntimeError("Could not locate state_dict in checkpoint.")\n\n\ndef load_model(path: Path, family: str, device):\n    obj = torch.load(path, map_location="cpu", weights_only=False)\n    if isinstance(obj, dict) and "class_order" in obj:\n        if tuple(obj["class_order"]) != CLASS_NAMES:\n            raise RuntimeError(f"Class order mismatch in {path}")\n\n    if family == "B0":\n        model = models.efficientnet_b0(weights=None)\n    elif family == "B3":\n        model = models.efficientnet_b3(weights=None)\n    else:\n        raise ValueError(family)\n\n    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)\n    state = normalize_state_keys(state_dict_from_checkpoint(obj))\n    model.load_state_dict(state, strict=True)\n    return model.to(device).eval(), obj\n\n\n@contextlib.contextmanager\ndef amp_context(device):\n    if device.type == "cuda":\n        with torch.autocast(device_type="cuda", dtype=torch.float16):\n            yield\n    else:\n        yield\n\n\n@torch.no_grad()\ndef predict_tta7(model, loader, device, desc):\n    model.eval()\n    ys, probs, ids, lesions = [], [], [], []\n\n    for x, y, image_ids, lesion_ids in tqdm(loader, desc=desc, leave=False):\n        x = x.to(device, non_blocking=True)\n        views = (\n            x,\n            torch.flip(x, dims=(3,)),\n            torch.flip(x, dims=(2,)),\n            torch.flip(x, dims=(2, 3)),\n            torch.rot90(x, 1, dims=(2, 3)),\n            torch.rot90(x, 2, dims=(2, 3)),\n            torch.rot90(x, 3, dims=(2, 3)),\n        )\n\n        total = None\n        for view in views:\n            with amp_context(device):\n                logits = model(view)\n            p = F.softmax(logits.float(), dim=1)\n            total = p if total is None else total + p\n        total = total / 7.0\n\n        ys.append(y.numpy())\n        probs.append(total.cpu().numpy())\n        ids.extend(list(image_ids))\n        lesions.extend(list(lesion_ids))\n\n    return np.concatenate(ys), np.concatenate(probs), ids, lesions\n\n\ndef core_metrics(y, probs):\n    y = np.asarray(y, dtype=int)\n    probs = np.asarray(probs, dtype=float)\n    pred = probs.argmax(axis=1)\n    return {\n        "accuracy": float(accuracy_score(y, pred)),\n        "macro_f1": float(\n            f1_score(\n                y,\n                pred,\n                labels=list(range(NUM_CLASSES)),\n                average="macro",\n                zero_division=0,\n            )\n        ),\n        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),\n        "melanoma_recall": float(\n            recall_score(y, pred, labels=[MEL_INDEX], average="macro", zero_division=0)\n        ),\n        "macro_auc": float(\n            roc_auc_score(\n                y,\n                probs,\n                labels=list(range(NUM_CLASSES)),\n                multi_class="ovr",\n                average="macro",\n            )\n        ),\n    }\n\n\ndef per_class_metrics(y, probs):\n    y = np.asarray(y, dtype=int)\n    probs = np.asarray(probs, dtype=float)\n    pred = probs.argmax(axis=1)\n    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y,\n        pred,\n        labels=list(range(NUM_CLASSES)),\n        zero_division=0,\n    )\n\n    rows = []\n    for i, c in enumerate(CLASS_NAMES):\n        tp = cm[i, i]\n        fn = cm[i, :].sum() - tp\n        fp = cm[:, i].sum() - tp\n        tn = cm.sum() - tp - fn - fp\n        specificity = float(tn / (tn + fp)) if (tn + fp) else float("nan")\n        binary = (y == i).astype(int)\n        rows.append({\n            "class_name": c,\n            "precision": float(precision[i]),\n            "recall": float(recall[i]),\n            "specificity": specificity,\n            "f1": float(f1[i]),\n            "support": int(support[i]),\n            "roc_auc": float(roc_auc_score(binary, probs[:, i])),\n            "average_precision": float(average_precision_score(binary, probs[:, i])),\n        })\n    return pd.DataFrame(rows), cm\n\n\ndef load_embedded_old_04b_predictions():\n    raw = gzip.decompress(base64.b64decode(OLD_04B_PRED_GZIP_B64.encode("ascii")))\n    if sha256_bytes(raw) != OLD_04B_PRED_SHA:\n        raise RuntimeError("Embedded historical LITE-GEN 04B prediction CSV hash mismatch.")\n    return pd.read_csv(io.BytesIO(raw))\n\n\ndef save_probabilities(path, ids, lesions, y, probs):\n    frame = pd.DataFrame({\n        "image_id": ids,\n        "lesion_id": lesions,\n        "true_label": y,\n        "true_class": [CLASS_NAMES[i] for i in y],\n    })\n    for i, c in enumerate(CLASS_NAMES):\n        frame[f"prob_{c}"] = probs[:, i]\n    frame.to_csv(path, index=False)\n    return frame\n\n\ndef save_final_predictions(path, ids, lesions, y, probs):\n    pred = probs.argmax(axis=1)\n    confidence = probs.max(axis=1)\n    entropy = -(np.clip(probs, 1e-12, 1.0) * np.log(np.clip(probs, 1e-12, 1.0))).sum(axis=1)\n    frame = pd.DataFrame({\n        "image_id": ids,\n        "lesion_id": lesions,\n        "true_label": y,\n        "true_class": [CLASS_NAMES[i] for i in y],\n        "pred_label": pred,\n        "pred_class": [CLASS_NAMES[i] for i in pred],\n        "correct": (pred == y).astype(int),\n        "max_probability": confidence,\n        "prediction_entropy": entropy,\n    })\n    for i, c in enumerate(CLASS_NAMES):\n        frame[f"prob_{c}"] = probs[:, i]\n    frame.to_csv(path, index=False)\n    return frame\n\n\ndef grouped_bootstrap(y, new_probs, old_probs, lesions):\n    y = np.asarray(y, dtype=int)\n    new_probs = np.asarray(new_probs, dtype=float)\n    old_probs = np.asarray(old_probs, dtype=float)\n    lesions = np.asarray(lesions, dtype=str)\n\n    unique = np.unique(lesions)\n    by_lesion = {g: np.flatnonzero(lesions == g) for g in unique}\n    rng = np.random.default_rng(BOOTSTRAP_SEED)\n\n    overall_rows = []\n    per_class_rows = []\n\n    for draw in tqdm(range(BOOTSTRAP_ITERS), desc="Lesion bootstrap 95% CI"):\n        sampled = rng.choice(unique, size=len(unique), replace=True)\n        idx = np.concatenate([by_lesion[g] for g in sampled])\n        yy = y[idx]\n\n        # Match the historical LITE-GEN 04B method.\n        if len(np.unique(yy)) < NUM_CLASSES:\n            continue\n\n        np_new = new_probs[idx]\n        np_old = old_probs[idx]\n        try:\n            mn = core_metrics(yy, np_new)\n            mo = core_metrics(yy, np_old)\n            pc_new, _ = per_class_metrics(yy, np_new)\n        except Exception:\n            continue\n\n        row = {"draw": draw, "n_images": int(len(idx))}\n        for metric in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"):\n            row[f"new_{metric}"] = mn[metric]\n            row[f"old04b_{metric}"] = mo[metric]\n            row[f"delta_{metric}"] = mn[metric] - mo[metric]\n        overall_rows.append(row)\n\n        for rec in pc_new.to_dict(orient="records"):\n            rec = dict(rec)\n            rec["draw"] = draw\n            per_class_rows.append(rec)\n\n    overall = pd.DataFrame(overall_rows)\n    per_class = pd.DataFrame(per_class_rows)\n    return overall, per_class\n\n\ndef summarize_bootstrap(overall, per_class, point_metrics, point_per_class):\n    metric_rows = []\n    for metric in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"):\n        new_vals = overall[f"new_{metric}"].to_numpy(float)\n        old_vals = overall[f"old04b_{metric}"].to_numpy(float)\n        delta_vals = overall[f"delta_{metric}"].to_numpy(float)\n        metric_rows.append({\n            "metric": metric,\n            "new_point": point_metrics[metric],\n            "new_ci95_lower": float(np.percentile(new_vals, 2.5)),\n            "new_ci95_upper": float(np.percentile(new_vals, 97.5)),\n            "old04b_point": OLD_04B_METRICS[metric],\n            "old04b_historical_ci95_lower": OLD_04B_CI[metric][0],\n            "old04b_historical_ci95_upper": OLD_04B_CI[metric][1],\n            "paired_delta_new_minus_old04b": point_metrics[metric] - OLD_04B_METRICS[metric],\n            "paired_delta_ci95_lower": float(np.percentile(delta_vals, 2.5)),\n            "paired_delta_ci95_upper": float(np.percentile(delta_vals, 97.5)),\n            "bootstrap_probability_delta_gt_0": float(np.mean(delta_vals > 0)),\n            "valid_iterations": int(len(delta_vals)),\n            "requested_iterations": BOOTSTRAP_ITERS,\n        })\n    metric_summary = pd.DataFrame(metric_rows)\n\n    pc_rows = []\n    for _, point in point_per_class.iterrows():\n        c = point["class_name"]\n        draws = per_class[per_class["class_name"].eq(c)]\n        for metric in ("precision", "recall", "specificity", "f1", "roc_auc", "average_precision"):\n            vals = draws[metric].to_numpy(float)\n            pc_rows.append({\n                "class_name": c,\n                "metric": metric,\n                "point": float(point[metric]),\n                "ci95_lower": float(np.percentile(vals, 2.5)),\n                "ci95_upper": float(np.percentile(vals, 97.5)),\n                "valid_iterations": int(len(vals)),\n            })\n    per_class_summary = pd.DataFrame(pc_rows)\n    return metric_summary, per_class_summary\n\n\ndef save_figure(fig, stem):\n    fig.tight_layout()\n    fig.savefig(FIG / f"{stem}.png", dpi=300, bbox_inches="tight")\n    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")\n    fig.savefig(FIG / f"{stem}.svg", bbox_inches="tight")\n    plt.close(fig)\n\n\ndef make_figures(test_df, y, probs, cm, per_class, metric_summary, delta_summary):\n    # 1. Class distribution\n    dist = (\n        test_df.groupby(["label", "class_name"], as_index=False)\n        .agg(images=("image_id", "size"), lesions=("lesion_id", "nunique"))\n        .sort_values("label")\n    )\n    dist.to_csv(SRC / "class_distribution.csv", index=False)\n    fig, ax = plt.subplots(figsize=(9, 5.5))\n    x = np.arange(len(dist))\n    ax.bar(x, dist["images"])\n    ax.set_xticks(x, dist["class_name"])\n    ax.set_ylabel("Images")\n    ax.set_title("Frozen clean TEST class distribution")\n    save_figure(fig, "FigS1_class_distribution")\n\n    # 2. Confusion matrices\n    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)\n    cm_df.to_csv(SRC / "confusion_matrix_counts.csv")\n    cm_norm = cm / cm.sum(axis=1, keepdims=True)\n    pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(\n        SRC / "confusion_matrix_row_normalized.csv"\n    )\n\n    for data, suffix, integer in (\n        (cm, "counts", True),\n        (cm_norm, "row_normalized", False),\n    ):\n        fig, ax = plt.subplots(figsize=(8, 7))\n        im = ax.imshow(data, aspect="auto")\n        ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")\n        ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)\n        ax.set_xlabel("Predicted class")\n        ax.set_ylabel("True class")\n        ax.set_title(f"Fixed B0-300 + B3-300 fusion: confusion matrix ({suffix})")\n        for i in range(NUM_CLASSES):\n            for j in range(NUM_CLASSES):\n                txt = f"{int(data[i, j])}" if integer else f"{data[i, j]:.2f}"\n                ax.text(j, i, txt, ha="center", va="center", fontsize=8)\n        fig.colorbar(im, ax=ax)\n        save_figure(\n            fig,\n            "Fig1_confusion_matrix_normalized" if not integer else "FigS2_confusion_matrix_counts",\n        )\n\n    # 3. ROC and PR source data + figures\n    roc_rows, pr_rows = [], []\n    fig_roc, ax_roc = plt.subplots(figsize=(8, 7))\n    fig_pr, ax_pr = plt.subplots(figsize=(8, 7))\n    for i, c in enumerate(CLASS_NAMES):\n        binary = (y == i).astype(int)\n        fpr, tpr, _ = roc_curve(binary, probs[:, i])\n        precision, recall, _ = precision_recall_curve(binary, probs[:, i])\n        auc = roc_auc_score(binary, probs[:, i])\n        ap = average_precision_score(binary, probs[:, i])\n\n        ax_roc.plot(fpr, tpr, label=f"{c} (AUC {auc:.3f})")\n        ax_pr.plot(recall, precision, label=f"{c} (AP {ap:.3f})")\n\n        for a, b in zip(fpr, tpr):\n            roc_rows.append({"class_name": c, "fpr": float(a), "tpr": float(b), "roc_auc": float(auc)})\n        for a, b in zip(recall, precision):\n            pr_rows.append({"class_name": c, "recall": float(a), "precision": float(b), "average_precision": float(ap)})\n\n    ax_roc.plot([0, 1], [0, 1], linestyle="--")\n    ax_roc.set_xlabel("False positive rate")\n    ax_roc.set_ylabel("True positive rate")\n    ax_roc.set_title("Multiclass one-vs-rest ROC curves")\n    ax_roc.legend(fontsize=8)\n    save_figure(fig_roc, "Fig2_multiclass_ROC")\n\n    ax_pr.set_xlabel("Recall")\n    ax_pr.set_ylabel("Precision")\n    ax_pr.set_title("Multiclass one-vs-rest precision-recall curves")\n    ax_pr.legend(fontsize=8)\n    save_figure(fig_pr, "Fig3_multiclass_precision_recall")\n\n    pd.DataFrame(roc_rows).to_csv(SRC / "roc_curve_data.csv", index=False)\n    pd.DataFrame(pr_rows).to_csv(SRC / "precision_recall_curve_data.csv", index=False)\n\n    # 4. Per-class performance\n    per_class.to_csv(SRC / "per_class_metrics_point.csv", index=False)\n    ordered = per_class.set_index("class_name").reindex(CLASS_NAMES)\n    x = np.arange(NUM_CLASSES)\n    width = 0.20\n    fig, ax = plt.subplots(figsize=(12, 6))\n    ax.bar(x - 1.5 * width, ordered["recall"], width, label="Recall")\n    ax.bar(x - 0.5 * width, ordered["f1"], width, label="F1")\n    ax.bar(x + 0.5 * width, ordered["roc_auc"], width, label="ROC-AUC")\n    ax.bar(x + 1.5 * width, ordered["average_precision"], width, label="AP")\n    ax.set_xticks(x, CLASS_NAMES)\n    ax.set_ylim(0, 1.0)\n    ax.set_ylabel("Score")\n    ax.set_title("Per-class performance of frozen fusion")\n    ax.legend()\n    save_figure(fig, "Fig4_per_class_performance")\n\n    # 5. Primary metric point + CI\n    ms = metric_summary.copy()\n    x = np.arange(len(ms))\n    yv = ms["new_point"].to_numpy(float)\n    lo = yv - ms["new_ci95_lower"].to_numpy(float)\n    hi = ms["new_ci95_upper"].to_numpy(float) - yv\n    fig, ax = plt.subplots(figsize=(10, 5.5))\n    ax.errorbar(x, yv, yerr=np.vstack([lo, hi]), fmt="o", capsize=5)\n    ax.set_xticks(x, [m.replace("_", "\\n") for m in ms["metric"]])\n    ax.set_ylim(0, 1.0)\n    ax.set_ylabel("Score")\n    ax.set_title("Frozen clean TEST primary metrics with lesion-bootstrap 95% CI")\n    save_figure(fig, "Fig5_primary_metrics_95CI")\n\n    # 6. LITE-GEN 04B vs new point comparison\n    width = 0.36\n    fig, ax = plt.subplots(figsize=(11, 6))\n    ax.bar(x - width / 2, ms["old04b_point"], width, label="LITE-GEN 04B")\n    ax.bar(x + width / 2, ms["new_point"], width, label="New fixed fusion")\n    ax.set_xticks(x, [m.replace("_", "\\n") for m in ms["metric"]])\n    ax.set_ylim(0, 1.0)\n    ax.set_ylabel("Score")\n    ax.set_title("Same frozen clean TEST: historical LITE-GEN 04B vs new fusion")\n    ax.legend()\n    save_figure(fig, "Fig6_LITEGEN04B_vs_new_fusion")\n\n    # 7. Paired delta CI\n    ds = delta_summary.copy()\n    xv = np.arange(len(ds))\n    point = ds["paired_delta_new_minus_old04b"].to_numpy(float)\n    dlo = point - ds["paired_delta_ci95_lower"].to_numpy(float)\n    dhi = ds["paired_delta_ci95_upper"].to_numpy(float) - point\n    fig, ax = plt.subplots(figsize=(10, 5.5))\n    ax.axhline(0.0, linestyle="--")\n    ax.errorbar(xv, point, yerr=np.vstack([dlo, dhi]), fmt="o", capsize=5)\n    ax.set_xticks(xv, [m.replace("_", "\\n") for m in ds["metric"]])\n    ax.set_ylabel("New - LITE-GEN 04B")\n    ax.set_title("Paired lesion-bootstrap performance difference")\n    save_figure(fig, "Fig7_paired_delta_vs_LITEGEN04B")\n\n\ndef save_error_summary(y, probs):\n    pred = probs.argmax(axis=1)\n    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))\n    rows = []\n    for i, true_c in enumerate(CLASS_NAMES):\n        total = cm[i, :].sum()\n        for j, pred_c in enumerate(CLASS_NAMES):\n            if i == j or cm[i, j] == 0:\n                continue\n            rows.append({\n                "true_class": true_c,\n                "predicted_class": pred_c,\n                "count": int(cm[i, j]),\n                "fraction_of_true_class": float(cm[i, j] / total) if total else float("nan"),\n            })\n    frame = pd.DataFrame(rows).sort_values(["count", "fraction_of_true_class"], ascending=False)\n    frame.to_csv(TAB / "table_error_confusion_pairs.csv", index=False)\n\n\ndef build_artifact_index():\n    main_files = {\n        "figures/Fig1_confusion_matrix_normalized.png",\n        "figures/Fig2_multiclass_ROC.png",\n        "figures/Fig3_multiclass_precision_recall.png",\n        "figures/Fig4_per_class_performance.png",\n        "figures/Fig5_primary_metrics_95CI.png",\n        "figures/Fig6_LITEGEN04B_vs_new_fusion.png",\n        "figures/Fig7_paired_delta_vs_LITEGEN04B.png",\n        "tables/table_primary_metrics_with_ci_and_comparison.csv",\n        "tables/table_per_class_metrics_with_ci.csv",\n    }\n    rows = []\n    for p in sorted(OUT.rglob("*")):\n        if not p.is_file():\n            continue\n        rel = str(p.relative_to(OUT))\n        if rel.startswith("figures/"):\n            kind = "figure"\n        elif rel.startswith("tables/"):\n            kind = "table"\n        elif rel.startswith("figure_source_data/"):\n            kind = "figure_source_data"\n        elif rel.startswith("audit/"):\n            kind = "audit"\n        elif "prediction" in rel or "probabilities" in rel:\n            kind = "prediction_data"\n        elif "bootstrap" in rel:\n            kind = "bootstrap_source"\n        else:\n            kind = "supporting_material"\n        rows.append({\n            "file": rel,\n            "kind": kind,\n            "placement": "main_text" if rel in main_files else "supplement_or_reproducibility",\n            "sha256": sha256_file(p),\n            "bytes": int(p.stat().st_size),\n        })\n    pd.DataFrame(rows).to_csv(OUT / "paper_material_index.csv", index=False)\n    (OUT / "paper_material_index.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")\n\n\ndef main():\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU.")\n\n    device = torch.device("cuda:0")\n    print("[GPU]", torch.cuda.get_device_name(0))\n    print("[CODE VERSION]", CODE_VERSION)\n    print("[STATUS] FROZEN MODEL. TEST REPORTING ONLY. NO TUNING.")\n\n    test_df, test_source = load_test_manifest()\n    required_cols = {"image_id", "lesion_id", "label", "class_name"}\n    if not required_cols.issubset(test_df.columns):\n        raise RuntimeError(f"Frozen TEST manifest missing columns: {sorted(required_cols - set(test_df.columns))}")\n    if len(test_df) != 971:\n        raise RuntimeError(f"Frozen TEST image count changed: {len(test_df)}")\n    if test_df["lesion_id"].nunique() != 749:\n        raise RuntimeError(f"Frozen TEST lesion count changed: {test_df[\'lesion_id\'].nunique()}")\n    if tuple(sorted(test_df["class_name"].unique())) != tuple(sorted(CLASS_NAMES)):\n        raise RuntimeError("Frozen TEST class set changed.")\n\n    old04b = load_embedded_old_04b_predictions()\n    if len(old04b) != 971:\n        raise RuntimeError("Embedded LITE-GEN 04B prediction row count changed.")\n\n    image_index = index_images(test_df)\n    checkpoints, provenance = find_locked_checkpoints()\n\n    audit_rows = []\n    for family, hashes in (("B0", B0_HASHES), ("B3", B3_HASHES)):\n        for seed, expected in hashes.items():\n            key = f"{family}_{seed}"\n            actual = sha256_file(checkpoints[key])\n            if actual != expected:\n                raise RuntimeError(f"Locked checkpoint hash mismatch: {key}")\n            audit_rows.append({\n                "family": family,\n                "seed": int(seed),\n                "expected_sha256": expected,\n                "actual_sha256": actual,\n                "source": provenance[key]["source"],\n            })\n    pd.DataFrame(audit_rows).to_csv(AUDIT / "checkpoint_hash_audit.csv", index=False)\n\n    env = {\n        "python": sys.version,\n        "platform": platform.platform(),\n        "torch": torch.__version__,\n        "torchvision": torchvision.__version__,\n        "numpy": np.__version__,\n        "pandas": pd.__version__,\n        "sklearn": sklearn.__version__,\n        "gpu": torch.cuda.get_device_name(0),\n    }\n    (AUDIT / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")\n\n    preflight = {\n        "experiment": "PAPER-EXP-01C",\n        "code_version": CODE_VERSION,\n        "test_manifest_source": test_source,\n        "test_manifest_sha256": TEST_MANIFEST_SHA,\n        "test_images": 971,\n        "test_lesions": 749,\n        "model_training_performed": False,\n        "checkpoint_selection_performed": False,\n        "tta_search_performed": False,\n        "ensemble_weight_search": False,\n        "threshold_search": False,\n        "calibration": False,\n        "fusion_locked_before_test": True,\n        "fusion": "0.50 * B0-300 equal3 TTA7 + 0.50 * B3-300 equal3 TTA7",\n        "historical_test_previously_observed_at_project_level": True,\n        "historical_04b_predictions_sha256": OLD_04B_PRED_SHA,\n    }\n    (AUDIT / "preflight_lock_audit.json").write_text(json.dumps(preflight, indent=2), encoding="utf-8")\n\n    loader = DataLoader(\n        TestDataset(test_df, image_index),\n        batch_size=BATCH_SIZE,\n        shuffle=False,\n        num_workers=NUM_WORKERS,\n        pin_memory=True,\n        persistent_workers=NUM_WORKERS > 0,\n        drop_last=False,\n    )\n\n    family_probs = {}\n    seed_outputs = {}\n    y_ref = ids_ref = lesions_ref = None\n\n    for family, hashes in (("B0", B0_HASHES), ("B3", B3_HASHES)):\n        probs_list = []\n        for seed in sorted(hashes):\n            key = f"{family}_{seed}"\n            model, ckpt_obj = load_model(checkpoints[key], family, device)\n            y, p, ids, lesions = predict_tta7(\n                model,\n                loader,\n                device,\n                desc=f"TEST {family} seed {seed} TTA7",\n            )\n            if y_ref is None:\n                y_ref, ids_ref, lesions_ref = y, ids, lesions\n            elif not np.array_equal(y_ref, y) or ids_ref != ids or lesions_ref != lesions:\n                raise RuntimeError("TEST row alignment changed between frozen checkpoints.")\n\n            if not np.isfinite(p).all() or not np.allclose(p.sum(axis=1), 1.0, atol=1e-5):\n                raise RuntimeError(f"Invalid probabilities: {key}")\n\n            probs_list.append(p)\n            seed_outputs[key] = p\n            save_probabilities(\n                OUT / f"{family.lower()}_seed_{seed}_test_probabilities.csv",\n                ids,\n                lesions,\n                y,\n                p,\n            )\n            del model\n            torch.cuda.empty_cache()\n\n        family_probs[family] = np.mean(probs_list, axis=0)\n        save_probabilities(\n            OUT / f"{family.lower()}_family_equal3_tta7_test_probabilities.csv",\n            ids_ref,\n            lesions_ref,\n            y_ref,\n            family_probs[family],\n        )\n\n    fusion = 0.5 * family_probs["B0"] + 0.5 * family_probs["B3"]\n    if not np.isfinite(fusion).all() or not np.allclose(fusion.sum(axis=1), 1.0, atol=1e-5):\n        raise RuntimeError("Final fixed fusion probabilities are invalid.")\n\n    new_predictions = save_final_predictions(\n        OUT / "paper_exp01c_frozen_fusion_test_predictions.csv",\n        ids_ref,\n        lesions_ref,\n        y_ref,\n        fusion,\n    )\n\n    # Exact alignment with the historical 04B prediction table.\n    old_map = old04b.set_index("image_id")\n    try:\n        aligned_old = old_map.loc[ids_ref].reset_index()\n    except KeyError as exc:\n        raise RuntimeError("Historical LITE-GEN 04B predictions do not align to frozen TEST.") from exc\n\n    if (\n        aligned_old["lesion_id"].astype(str).tolist() != [str(x) for x in lesions_ref]\n        or not np.array_equal(aligned_old["true_label"].to_numpy(int), y_ref)\n    ):\n        raise RuntimeError("Historical 04B labels/lesions do not align with frozen TEST.")\n\n    old_probs = aligned_old[PROB_COLS].to_numpy(float)\n    aligned_old.to_csv(OUT / "historical_litegen04b_predictions_for_paired_comparison.csv", index=False)\n\n    metrics = core_metrics(y_ref, fusion)\n    pc, cm = per_class_metrics(y_ref, fusion)\n\n    # Bootstrap same lesion sampling for both systems and their paired differences.\n    boot, pc_boot = grouped_bootstrap(y_ref, fusion, old_probs, lesions_ref)\n    boot.to_csv(OUT / "bootstrap_primary_metrics_draws.csv", index=False)\n    pc_boot.to_csv(OUT / "bootstrap_per_class_draws.csv", index=False)\n\n    metric_summary, pc_ci = summarize_bootstrap(boot, pc_boot, metrics, pc)\n    metric_summary.to_csv(TAB / "table_primary_metrics_with_ci_and_comparison.csv", index=False)\n    pc_ci.to_csv(TAB / "table_per_class_metrics_with_ci.csv", index=False)\n    pc.to_csv(TAB / "table_per_class_metrics_point.csv", index=False)\n\n    # Convenience wide per-class paper table.\n    ci_wide = pc_ci.pivot(index="class_name", columns="metric", values=["ci95_lower", "ci95_upper"])\n    ci_wide.columns = [f"{a}_{b}" for a, b in ci_wide.columns]\n    wide = pc.set_index("class_name").join(ci_wide).reset_index()\n    wide.to_csv(TAB / "table_per_class_full_with_ci.csv", index=False)\n\n    # Same-test point comparison.\n    comparison_rows = []\n    for metric in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"):\n        comparison_rows.append({\n            "metric": metric,\n            "litegen04b": OLD_04B_METRICS[metric],\n            "new_fixed_fusion": metrics[metric],\n            "delta_new_minus_litegen04b": metrics[metric] - OLD_04B_METRICS[metric],\n        })\n    pd.DataFrame(comparison_rows).to_csv(TAB / "table_litegen04b_vs_new_fusion_same_test.csv", index=False)\n\n    save_error_summary(y_ref, fusion)\n    make_figures(test_df, y_ref, fusion, cm, pc, metric_summary, metric_summary)\n\n    class_counts = (\n        test_df.groupby(["label", "class_name"], as_index=False)\n        .agg(images=("image_id", "size"), lesions=("lesion_id", "nunique"))\n        .sort_values("label")\n    )\n    class_counts.to_csv(TAB / "table_test_class_distribution.csv", index=False)\n\n    summary = {\n        "experiment": "PAPER-EXP-01C",\n        "code_version": CODE_VERSION,\n        "scientific_status": (\n            "frozen repeated clean internal TEST benchmark; TEST historically observed "\n            "by LITE-GEN 04B at project level"\n        ),\n        "model_frozen_before_test": True,\n        "locked_system": {\n            "resolution": 300,\n            "b0_checkpoints": B0_HASHES,\n            "b3_checkpoints": B3_HASHES,\n            "b0_inference": "TTA7 then equal 1/3 probability average",\n            "b3_inference": "TTA7 then equal 1/3 probability average",\n            "family_fusion": "fixed 0.50 B0 + 0.50 B3 probability average",\n            "threshold": "argmax",\n            "calibration": "none",\n        },\n        "test_manifest_sha256": TEST_MANIFEST_SHA,\n        "test_images": int(len(test_df)),\n        "test_lesions": int(test_df["lesion_id"].nunique()),\n        "test_class_image_counts": {\n            row["class_name"]: int(row["images"])\n            for _, row in class_counts.iterrows()\n        },\n        "primary_new_fusion_test_metrics": metrics,\n        "new_fusion_lesion_bootstrap_95ci": {\n            row["metric"]: {\n                "lower": float(row["new_ci95_lower"]),\n                "upper": float(row["new_ci95_upper"]),\n                "valid_iterations": int(row["valid_iterations"]),\n            }\n            for _, row in metric_summary.iterrows()\n        },\n        "historical_litegen04b_same_test_metrics": OLD_04B_METRICS,\n        "paired_new_minus_litegen04b": {\n            row["metric"]: {\n                "point_delta": float(row["paired_delta_new_minus_old04b"]),\n                "ci95_lower": float(row["paired_delta_ci95_lower"]),\n                "ci95_upper": float(row["paired_delta_ci95_upper"]),\n                "bootstrap_probability_delta_gt_0": float(row["bootstrap_probability_delta_gt_0"]),\n            }\n            for _, row in metric_summary.iterrows()\n        },\n        "bootstrap_iterations_requested": BOOTSTRAP_ITERS,\n        "bootstrap_valid_iterations": int(len(boot)),\n        "bootstrap_seed": BOOTSTRAP_SEED,\n        "model_training_performed": False,\n        "external_data_used": False,\n        "ensemble_weight_search": False,\n        "threshold_search": False,\n        "calibration": False,\n        "post_test_model_selection": False,\n        "post_test_model_modification_allowed": False,\n        "decision": "REPORT_FIXED_FUSION_EXACTLY_AS_OBSERVED_NO_FURTHER_CLEAN_TUNING",\n    }\n    (OUT / "paper_exp01c_final_summary.json").write_text(\n        json.dumps(summary, indent=2), encoding="utf-8"\n    )\n\n    manuscript_text = f"""PAPER-EXP-01C — FROZEN CLEAN INTERNAL TEST\n\nSystem:\nFixed 50/50 probability fusion of the accepted B0-300 and B3-300 development families.\nEach family consists of three independently trained seeds, evaluated with the preselected\nTTA7 rule and averaged equally within family.\n\nScientific status:\nThis is a frozen repeated clean internal TEST benchmark. The same 971-image / 749-lesion\nTEST partition was historically observed by LITE-GEN 04B; therefore it must not be called\na pristine never-seen project-level test.\n\nPrimary new fusion TEST metrics:\nAccuracy: {metrics[\'accuracy\']:.6f}\nMacro-F1: {metrics[\'macro_f1\']:.6f}\nBalanced accuracy: {metrics[\'balanced_accuracy\']:.6f}\nMelanoma recall: {metrics[\'melanoma_recall\']:.6f}\nMacro AUC: {metrics[\'macro_auc\']:.6f}\n\nHistorical LITE-GEN 04B same-TEST reference:\nAccuracy: {OLD_04B_METRICS[\'accuracy\']:.6f}\nMacro-F1: {OLD_04B_METRICS[\'macro_f1\']:.6f}\nBalanced accuracy: {OLD_04B_METRICS[\'balanced_accuracy\']:.6f}\nMelanoma recall: {OLD_04B_METRICS[\'melanoma_recall\']:.6f}\nMacro AUC: {OLD_04B_METRICS[\'macro_auc\']:.6f}\n\nNo model, TTA, weight, threshold or calibration change is permitted after observing\nthis result. Report the fixed system exactly as evaluated.\n"""\n    (OUT / "manuscript_result_block.txt").write_text(manuscript_text, encoding="utf-8")\n\n    # Preserve protocol and code snapshot inside the result archive.\n    protocol_runtime = {\'experiment\': \'PAPER-EXP-01C\', \'title\': \'Frozen Clean Internal Test of Fixed B0-300 + B3-300 Fusion\', \'code_version\': \'PAPEREXP01C_FROZEN_CLEAN_INTERNAL_TEST_FIXED_B0B3_FUSION_2026-08-23_v1\', \'scientific_status\': \'Frozen repeated clean internal TEST benchmark. The 971-image/749-lesion TEST was historically observed by LITE-GEN 04B, so this is not described as a pristine never-seen project-level test.\', \'model_lock_completed_before_this_test\': True, \'selection_basis\': \'frozen lesion-safe TRAIN/VALIDATION only\', \'test_manifest\': {\'images\': 971, \'lesions\': 749, \'sha256\': \'5c6f3cdca8ae60e200d6e5f5f1d619b3f08df71a2dadab95cdad7948a2834eec\'}, \'classes\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\', \'DF\', \'VASC\'], \'locked_system\': {\'input_resolution\': 300, \'b0_family\': {\'architecture\': \'3 x Torchvision EfficientNet-B0\', \'checkpoint_sha256\': {\'20260820\': \'30efcb4f88cfddd4f6ece227d920fbab27caf38e821f2e442941510b199bfb39\', \'20260821\': \'a79eef838354a6613d75b7aa455d643ccd0f3309b23a31f177c3169f723575cb\', \'20260822\': \'f258083d9bb36858ad96ce2d2fd8ba54a0834dc5b8f38dcbb1ee762783509587\'}, \'per_seed_inference\': \'TTA7 fixed geometric average\', \'within_family_ensemble\': \'equal 1/3 probability average\'}, \'b3_family\': {\'architecture\': \'3 x Torchvision EfficientNet-B3\', \'checkpoint_sha256\': {\'20260820\': \'59a23d41c48cb1ccbf9184e9cf67c115717775560283bb4a1da95f14698dc296\', \'20260821\': \'723176d2c08b759b9a8c2cbbfe245840ad844f1a77da748948c75c09afc51d69\', \'20260822\': \'2dc2cf4d28b6f5b8ccdb61a173aff25ba80ab2e0c794dfdfafcb104a87d89a4d\'}, \'per_seed_inference\': \'TTA7 fixed geometric average\', \'within_family_ensemble\': \'equal 1/3 probability average\'}, \'between_family_fusion\': \'fixed 50% B0-family probability + 50% B3-family probability\', \'tta7_views\': [\'original\', \'horizontal flip\', \'vertical flip\', \'horizontal+vertical flip\', \'rotation 90 degrees\', \'rotation 180 degrees\', \'rotation 270 degrees\'], \'thresholds\': \'argmax only\', \'calibration\': \'none\', \'weight_search\': \'none\'}, \'validation_lock_evidence\': {\'fixed_fusion_accuracy\': 0.8539865513928915, \'fixed_fusion_macro_f1\': 0.7893572669914984, \'fixed_fusion_balanced_accuracy\': 0.7602718482018861, \'fixed_fusion_melanoma_recall\': 0.5520833333333334, \'fixed_fusion_macro_auc\': 0.9652373431392501}, \'historical_litegen04b_reference\': {\'accuracy\': 0.8527291452111225, \'macro_f1\': 0.6713432549644158, \'balanced_accuracy\': 0.6912816493603928, \'melanoma_recall\': 0.638095238095238, \'macro_auc\': 0.9696825962107176, \'prediction_csv_sha256\': \'5a2c5ca2a39319bec601d1fb6e999c21b909f6f829a9bb76a82643104c1b0a4a\', \'role\': \'paired historical comparator only; not used to tune the new fusion\'}, \'bootstrap\': {\'unit\': \'lesion\', \'requested_iterations\': 2000, \'seed\': 20260821, \'rule\': \'skip resamples missing any of the seven classes, matching LITE-GEN 04B reporting\'}, \'hard_rules\': [\'No model training.\', \'No checkpoint selection.\', \'No TTA selection.\', \'No ensemble-weight search.\', \'No threshold search.\', \'No calibration.\', \'No post-test model modification.\', \'Report the locked 50/50 fusion exactly as observed.\', \'Do not use this TEST result to launch another clean-development search.\']}\n    (OUT / "paper_exp01c_protocol.json").write_text(\n        json.dumps(protocol_runtime, indent=2), encoding="utf-8"\n    )\n    try:\n        shutil.copy2(Path(__file__), OUT / "paper_exp01c_code_snapshot.py")\n    except Exception:\n        pass\n\n    build_artifact_index()\n\n    results_zip = WORKING_ROOT / "PAPEREXP01C_Frozen_Clean_Internal_Test_Results.zip"\n    with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(OUT.rglob("*")):\n            if not p.is_file():\n                continue\n            # Do not copy extracted checkpoints into the Results package.\n            if "locked_checkpoints" in p.parts or p.suffix.lower() in {".pth", ".pt", ".ckpt"}:\n                continue\n            z.write(p, arcname=str(p.relative_to(OUT)))\n\n    print("\\n" + "=" * 112)\n    print("PAPER-EXP-01C COMPLETE — FROZEN CLEAN INTERNAL TEST")\n    print("=" * 112)\n    print(json.dumps(metrics, indent=2))\n    print("\\nHistorical LITE-GEN 04B same TEST:")\n    print(json.dumps(OLD_04B_METRICS, indent=2))\n    print("\\nPaired comparison:")\n    print(metric_summary[\n        [\n            "metric",\n            "paired_delta_new_minus_old04b",\n            "paired_delta_ci95_lower",\n            "paired_delta_ci95_upper",\n            "bootstrap_probability_delta_gt_0",\n        ]\n    ].to_string(index=False))\n    print("\\nVALID BOOTSTRAP DRAWS:", len(boot), "/", BOOTSTRAP_ITERS)\n    print("POST-TEST TUNING ALLOWED:", False)\n    print("\\nUPLOAD THIS FILE:")\n    print(results_zip)\n    print("SHA256:", sha256_file(results_zip))\n\n\nif __name__ == "__main__":\n    main()\n'
p = Path("/kaggle/working/PAPEREXP01C_frozen_clean_internal_test_fixed_fusion.py")
p.write_text(SCRIPT, encoding="utf-8")
print("Wrote:", p)
print("Code version:", 'PAPEREXP01C_FROZEN_CLEAN_INTERNAL_TEST_FIXED_B0B3_FUSION_2026-08-23_v1')
print("Script bytes:", p.stat().st_size)


Wrote: /kaggle/working/PAPEREXP01C_frozen_clean_internal_test_fixed_fusion.py
Code version: PAPEREXP01C_FROZEN_CLEAN_INTERNAL_TEST_FIXED_B0B3_FUSION_2026-08-23_v1
Script bytes: 99003


In [2]:
import subprocess, sys

cmd = [
    sys.executable,
    "-u",
    "/kaggle/working/PAPEREXP01C_frozen_clean_internal_test_fixed_fusion.py",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /usr/bin/python3 -u /kaggle/working/PAPEREXP01C_frozen_clean_internal_test_fixed_fusion.py
[GPU] Tesla T4
[CODE VERSION] PAPEREXP01C_FROZEN_CLEAN_INTERNAL_TEST_FIXED_B0B3_FUSION_2026-08-23_v1
[STATUS] FROZEN MODEL. TEST REPORTING ONLY. NO TUNING.
[TEST MANIFEST] /kaggle/input/datasets/jafarsadat/litegen03-clean-split/litegen03_test_manifest.csv


Lesion bootstrap 95% CI: 100%|██████████| 2000/2000 [01:21<00:00, 24.63it/s]



PAPER-EXP-01C COMPLETE — FROZEN CLEAN INTERNAL TEST
{
  "accuracy": 0.8681771369721936,
  "macro_f1": 0.7333458465858439,
  "balanced_accuracy": 0.7076330978711177,
  "melanoma_recall": 0.7523809523809524,
  "macro_auc": 0.9756287619152376
}

Historical LITE-GEN 04B same TEST:
{
  "accuracy": 0.8527291452111225,
  "macro_f1": 0.6713432549644158,
  "balanced_accuracy": 0.6912816493603928,
  "melanoma_recall": 0.638095238095238,
  "macro_auc": 0.9696825962107176
}

Paired comparison:
           metric  paired_delta_new_minus_old04b  paired_delta_ci95_lower  paired_delta_ci95_upper  bootstrap_probability_delta_gt_0
         accuracy                       0.015448                -0.003061                 0.035343                          0.940733
         macro_f1                       0.062003                 0.001681                 0.133810                          0.978905
balanced_accuracy                       0.016351                -0.034466                 0.085670               

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/working/PAPEREXP01C_frozen_clean_internal_test_fixed_fusion.py'], returncode=0)

In [3]:
from pathlib import Path
import hashlib

p = Path("/kaggle/working/PAPEREXP01C_Frozen_Clean_Internal_Test_Results.zip")
print("\nUPLOAD THIS FILE:")
print(p)
print("Exists:", p.exists())
if p.exists():
    print("Size:", p.stat().st_size)
    print("SHA256:", hashlib.sha256(p.read_bytes()).hexdigest())



UPLOAD THIS FILE:
/kaggle/working/PAPEREXP01C_Frozen_Clean_Internal_Test_Results.zip
Exists: True
Size: 2443753
SHA256: d0212cc9f22d111a554ffb42894f3809162b59b6eb370c97955d8303022182a3
